In [1]:
import os

# ১. পুরোনো কনফিগারেশন মুছে ফেলা
!rm -rf ~/.kaggle/kaggle.json

# ২. নতুন kaggle.json কপি ও পারমিশন সেট করা
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ৩. টেস্ট করুন API ঠিকমতো কাজ করছে কিনা
!kaggle datasets list -s plantvillage

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [2]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

!unzip -q plantvillage-dataset.zip -d plant_village_dataset

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:23<00:00, 94.8MB/s]



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# ১. ডেটা ট্রান্সফর্মেশন ও অগমেন্টেশন (Data Augmentation & Normalization)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# ২. ডেটা লোড করার ফাংশন (Folder Structure: dataset/train/ and dataset/val/)
# ২. ডেটা লোড করার সঠিক পাথ নির্ধারণ
data_dir = 'plant_village_dataset/plantvillage dataset/color'

# পুরো ডেটাসেট একসাথে লোড করা
full_dataset = datasets.ImageFolder(data_dir, transform=data_transforms['train'])

# ডেটাসেটকে ৮০% Train এবং ২০% Validation এ ভাগ করা
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# DataLoader তৈরি করা (num_workers=0 দিয়ে নিরাপদে রান হবে)
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
}

image_datasets = {'train': train_dataset, 'val': val_dataset}
class_names = full_dataset.classes
num_classes = len(class_names)

print(f"Total Classes Identified: {num_classes}")

# ৩. প্রি-ট্রেইনড মডেল লোড এবং কাস্টমাইজেশন (Transfer Learning - ResNet18)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet18(pretrained=True)

# ResNet-এর লাস্ট লেয়ার পরিবর্তন করে আমাদের ক্লাস সংখ্যার সাথে ম্যাচ করানো
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

# ৪. লস ফাংশন এবং অপটিমাইজার নির্ধারণ
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ৫. মডেল ট্রেনিং ফাংশন
def train_model(model, criterion, optimizer, num_epochs=5):
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    return model

# ৬. ট্রেনিং শুরু করা (কম সময়ে চেকের জন্য Epochs = 3 বা 5 রাখুন)
trained_model = train_model(model, criterion, optimizer, num_epochs=5)

# ৭. মডেল সেভ করা (ভবিষ্যতে ড্যাশবোর্ডে ব্যবহারের জন্য)
torch.save(trained_model.state_dict(), 'crop_disease_model.pth')
print("Model trained and saved successfully!")

Total Classes Identified: 38
Using device: cuda:0
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


Epoch 1/5
----------
Train Loss: 0.3275 Acc: 0.8989
Val Loss: 0.1708 Acc: 0.9424
Epoch 2/5
----------
Train Loss: 0.1448 Acc: 0.9522
Val Loss: 0.0831 Acc: 0.9704
Epoch 3/5
----------
Train Loss: 0.1068 Acc: 0.9652
Val Loss: 0.1303 Acc: 0.9587
Epoch 4/5
----------
Train Loss: 0.0882 Acc: 0.9720
Val Loss: 0.1906 Acc: 0.9517
Epoch 5/5
----------
Train Loss: 0.0770 Acc: 0.9749
Val Loss: 0.1175 Acc: 0.9634
Model trained and saved successfully!


In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from google.colab import files

# ১. ছবি আপলোড করার সুবিধা
print("গাছের পাতার ছবি আপলোড করুন:")
uploaded = files.upload()

# ২. আপলোড করা ছবিটির প্রসেসিং
for filename in uploaded.keys():
    # ছবি ওপেন করা
    image = Image.open(filename).convert('RGB')

    # ছবিটির সাইজ ও ফরম্যাট মডেলের উপযোগী করা
    transform = data_transforms['val']
    input_tensor = transform(image).unsqueeze(0).to(device)

    # ৩. মডেল দিয়ে প্রেডিকশন করা
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        _, predicted_class = torch.max(output, 1)

    # ফলাফল দেখানো
    predicted_label = class_names[predicted_class.item()]
    print("\n-------------------------------------------")
    print(f"📸 ছবির নাম: {filename}")
    print(f"🔍 শনাক্তকৃত রোগ/অবস্থা: {predicted_label}")
    print("-------------------------------------------")

গাছের পাতার ছবি আপলোড করুন:


In [ ]:
import torch
torch.save(model.state_dict(), 'plant_disease_resnet18.pth')